![](https://upload.wikimedia.org/wikipedia/commons/4/42/UNIR_Horizontal_Logo.png)


#**Tema 2. El texto como dato.**
# Caso de uso: Servicio técnico para alumnos de UNIR


En esten notebook vamos a utilizar un mismo mensaje con spaCy, NLTK y un modelo de IA generativa.
Algunos conceptos: spaCy y NLTK son bibliotecas,  `es_core_news_sm` es el modelo español que usaremos en este notebook.

## Preparación

In [2]:
INSTALAR = True
if INSTALAR:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'spacy==3.8.7', 'nltk==3.9.2', 'tokenizers==0.22.1',
      'https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl'], check=True)
else:
    print('Instalación omitida. Ejecuta con INSTALAR=True si faltan las bibliotecas.')

## 2 · Texto original y desafío de negación
El caso pide recuperar acceso. La alumna **niega** que quiera anular la matrícula. Ni la urgencia ni la causa técnica constan en el mensaje.

In [7]:
# Herramientas que utilizaremos:
import re    # Para extraer identificadores como INC-204.
import json  # Para trabajar con la respuesta JSON de la IA generativa.

from IPython.display import display, HTML  # Para visualizar resultados.


# ------------------------------------------------------------
# 1. MENSAJE PRINCIPAL: incidencia ficticia de una alumna de UNIR
# ------------------------------------------------------------

# Las dos cadenas entre paréntesis se unen automáticamente.
TEXTO = (
    'Hola, soy Ana, alumna de UNIR en Madrid. Desde ayer no puedo acceder al campus virtual. '
    'No quiero anular mi matrícula, solo recuperar el acceso. Ticket INC-204.'
)

# ¿Qué información queremos identificar?
# - Persona: Ana.
# - Organización: UNIR.
# - Lugar: Madrid.
# - Problema: no puede acceder al campus virtual.
# - Intención: recuperar el acceso.
# - Negación: NO quiere anular su matrícula.
# - Ticket: INC-204.
#
# El mensaje no indica la causa técnica ni la urgencia.
# No debemos inventar esa información.


# ------------------------------------------------------------
# 2. CORPUS: colección de mensajes del servicio técnico
# ------------------------------------------------------------

CORPUS = [
    # Caso 1: recuperar el acceso, sin anular la matrícula.
    TEXTO,

    # Caso 2: problema al entregar una actividad.
    'No puedo entregar una actividad en el campus virtual de UNIR. Ticket INC-205.',

    # Caso 3: solicita anular la matrícula y afirma que puede acceder.
    'Quiero anular mi matrícula. Puedo acceder al campus sin problemas. Ticket INC-206.'
]

# Los casos permiten comparar problemas e intenciones diferentes.
# Encontrar la palabra "anular" no basta para saber qué quiere el alumno:
# debemos distinguir "quiero anular" de "no quiero anular".


# ------------------------------------------------------------
# 3. MOSTRAMOS EL MENSAJE QUE ANALIZAREMOS
# ------------------------------------------------------------

# Utilizaremos este mismo texto con spaCy, NLTK y una IA generativa.
# Este bloque solo prepara los datos; todavía no realiza ningún análisis.
print(TEXTO)

Hola, soy Ana, alumna de UNIR en Madrid. Desde ayer no puedo acceder al campus virtual. No quiero anular mi matrícula, solo recuperar el acceso. Ticket INC-204.


## 3 · spaCy
`nlp(texto)` tokeniza y ejecuta componentes ya entrenados. La tabla añade información al texto sin borrarlo. `lemma_` es el lema, `pos_` la categoría, `morph` los rasgos e `is_stop` una marca.

**Referencia lingüística:** “Ayer vino Ana” → venir, VERB, pretérito perfecto simple, tercera persona singular. “Ana compró vino” → vino, NOUN, bebida. La referencia humana y la predicción se comparan, no se equiparan automáticamente.

In [9]:
import spacy
nlp = spacy.load('es_core_news_sm')
print('spaCy', spacy.__version__, 'modelo', nlp.meta['version'])
doc = nlp(TEXTO)
filas = [{'token':t.text,'lema':t.lemma_,'POS':t.pos_,
          'rasgos':str(t.morph),'stopword':t.is_stop} for t in doc]
try:
    import pandas as pd
    display(pd.DataFrame(filas))
except ImportError:
    for fila in filas: print(fila)
for frase in ['Ayer vino Ana.', 'Ana compró vino.']:
    print(frase, [(t.text,t.lemma_,t.pos_,str(t.morph)) for t in nlp(frase) if t.lower_=='vino'])
print('El original permanece:', doc.text == TEXTO)

spaCy 3.8.7 modelo 3.8.0


,token,lema,POS,rasgos,stopword
0,Hola,Hola,PROPN,,False
1,",",",",PUNCT,PunctType=Comm,False
2,soy,soy,PROPN,,True
3,Ana,Ana,PROPN,,False
4,",",",",PUNCT,PunctType=Comm,False
5,alumna,alumna,NOUN,Gender=Fem|Number=Sing,False
6,de,de,ADP,,True
7,UNIR,UNIR,PROPN,,False
8,en,en,ADP,,True
9,Madrid,Madrid,PROPN,,False


Ayer vino Ana. [('vino', 'venir', 'VERB', 'Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin')]
Ana compró vino. [('vino', 'vino', 'NOUN', 'Gender=Masc|Number=Sing')]
El original permanece: True


## 4 · Entidades del modelo y reglas del dominio
Primero observa **lo que devuelve realmente** el modelo genérico. Puede no reconocer UNIR como organización o el ticket como entidad. Eso no invalida toda la herramienta.

Después añadimos reglas explícitas: `UNIR` como organización, `campus virtual` como servicio y `INC-204` como ticket. Son decisiones que programamos, no descubrimientos autónomos del modelo. La expresión regular del ticket es adecuada para este formato sintético, no para todos los identificadores posibles.

In [10]:
# Este bloque parte de:
# - TEXTO: mensaje de la alumna.
# - doc: resultado de procesar TEXTO con el modelo genérico de spaCy.
# - Las importaciones previas de spacy, re, display y HTML.


# ------------------------------------------------------------
# 1. MOSTRAR LAS ENTIDADES DETECTADAS POR EL MODELO GENÉRICO
# ------------------------------------------------------------

# doc.ents contiene los fragmentos que el modelo reconoce como entidades.
# Para cada entidad:
# - e.text: texto detectado, por ejemplo "Ana" o "UNIR".
# - e.label_: categoría asignada, por ejemplo PER, ORG o LOC.
#
# Son predicciones del modelo: pueden contener errores.
print(
    'Entidades del modelo genérico:',
    [(e.text, e.label_) for e in doc.ents]
)

# displaCy es el visualizador de spaCy.
# style='ent': resalta las entidades y muestra sus etiquetas.
# jupyter=False: devuelve el HTML para que lo mostremos explícitamente.
# HTML(...) y display(...) permiten visualizarlo dentro del notebook.
display(
    HTML(
        spacy.displacy.render(doc, style='ent', jupyter=False)
    )
)


# ------------------------------------------------------------
# 2. EXTRAER EL IDENTIFICADOR DEL TICKET CON UNA REGLA DE FORMATO
# ------------------------------------------------------------

# Buscamos un identificador como INC-204:
# - \b: límite de palabra.
# - INC-: texto literal que debe aparecer.
# - \d+: uno o más dígitos.
# - \b: límite de palabra al terminar.
#
# re.search devuelve la primera coincidencia o None si no encuentra nada.
# Esta regla reconoce un formato; no necesita un modelo lingüístico.
ticket = re.search(r'\bINC-\d+\b', TEXTO)

# group(0) devuelve el texto completo de la coincidencia: "INC-204".
# Solo lo consultamos si se ha encontrado un ticket.
print('Ticket por regex:', ticket.group(0) if ticket else None)


# ------------------------------------------------------------
# 3. CREAR OTRO PIPELINE PARA AÑADIR REGLAS DEL DOMINIO UNIR
# ------------------------------------------------------------

# Cargamos una segunda instancia del modelo.
# Así podemos comparar el modelo genérico con el modelo más nuestras reglas,
# conservando los resultados originales en doc.
nlp_dominio = spacy.load('es_core_news_sm')

# Añadimos EntityRuler: un componente que reconoce entidades mediante
# patrones definidos por nosotros.
#
# after='ner':
#   Ejecuta las reglas después del reconocedor de entidades del modelo.
#
# overwrite_ents=True:
#   Permite que nuestras coincidencias sustituyan entidades del modelo
#   cuando se solapan. No borra todas las entidades anteriores.
#
# phrase_matcher_attr='LOWER':
#   Compara los patrones de texto ignorando mayúsculas y minúsculas.
#   Por ejemplo, "UNIR" y "unir" pueden coincidir con la misma regla.
ruler = nlp_dominio.add_pipe(
    'entity_ruler',
    after='ner',
    config={
        'overwrite_ents': True,
        'phrase_matcher_attr': 'LOWER'
    }
)

# Definimos dos convenciones para nuestro caso de soporte:
# - "UNIR" se etiqueta como organización.
# - "campus virtual" se etiqueta como servicio.
#
# SERVICIO es una etiqueta propia que introducimos en esta demo.
# Añadir estas reglas no entrena el modelo: incorpora patrones explícitos.
ruler.add_patterns([
    {'label': 'ORG', 'pattern': 'UNIR'},
    {'label': 'SERVICIO', 'pattern': 'campus virtual'}
])


# ------------------------------------------------------------
# 4. PROCESAR EL MISMO MENSAJE CON EL MODELO Y LAS REGLAS
# ------------------------------------------------------------

# El texto pasa por los componentes del modelo y después por EntityRuler.
# El resultado combina predicciones del modelo con nuestras reglas.
doc_dominio = nlp_dominio(TEXTO)

print(
    'Modelo más reglas:',
    [(e.text, e.label_) for e in doc_dominio.ents]
)

# Estas reglas solo actúan sobre los patrones indicados.
# No corrigen automáticamente otros errores del modelo.
# Si el modelo ya reconocía UNIR como ORG, esa etiqueta seguirá igual.


# ------------------------------------------------------------
# 5. DISTINGUIR EXTRACCIÓN DE DATOS E INTERPRETACIÓN DE INTENCIÓN
# ------------------------------------------------------------

# El ticket se ha extraído aparte con re.search.
# No lo hemos añadido como entidad a doc_dominio.ents.
print(
    'La regla del ticket se mantiene separada:',
    ticket.group(0) if ticket else None
)

# Identificar personas, organizaciones, servicios y tickets
# no resuelve por sí solo qué solicita la alumna.
#
# Para distinguir "quiero anular" de "no quiero anular" necesitamos
# un componente adicional que analice o clasifique la intención.


Entidades del modelo genérico: [('Hola', 'PER'), ('Ana', 'PER'), ('UNIR', 'ORG'), ('Madrid', 'LOC'), ('No quiero anular mi matrícula', 'MISC'), ('Ticket INC-204', 'MISC')]


Ticket por regex: INC-204
Modelo más reglas: [('Hola', 'PER'), ('Ana', 'PER'), ('UNIR', 'ORG'), ('Madrid', 'LOC'), ('campus virtual', 'SERVICIO'), ('No quiero anular mi matrícula', 'MISC'), ('Ticket INC-204', 'MISC')]
La regla del ticket se mantiene separada: INC-204


## 5 · Negación y decisiones de representación
`is_stop` no elimina tokens. El siguiente código sí los filtra y además lematiza. Proteger negaciones ayuda, pero no conserva automáticamente el alcance, el orden ni todos los matices.

Observa que “no quiero anular” tiene una intención opuesta a “quiero anular”. Un conteo de la palabra “anular” no resuelve esa diferencia.

In [12]:
NEGACIONES={'no','ni','nunca','jamás','tampoco','sin'}
def representar(t,proteger=False):
    return [x.lemma_.lower() for x in nlp(t)
            if not x.is_punct and not x.is_space
            and (not x.is_stop or (proteger and x.lower_ in NEGACIONES))]
for t in ['No quiero anular mi matrícula.', 'Quiero anular mi matrícula.',
          'No puedo acceder al campus virtual.']:
    print('\nOriginal:',t)
    print('Quitar stopwords y lematizar:',representar(t))
    print('Proteger negaciones:',representar(t,True))


Original: No quiero anular mi matrícula.
Quitar stopwords y lematizar: ['querer', 'anular', 'matrícula']
Proteger negaciones: ['no', 'querer', 'anular', 'matrícula']

Original: Quiero anular mi matrícula.
Quitar stopwords y lematizar: ['querer', 'anular', 'matrícula']
Proteger negaciones: ['querer', 'anular', 'matrícula']

Original: No puedo acceder al campus virtual.
Quitar stopwords y lematizar: ['acceder', 'campus', 'virtual']
Proteger negaciones: ['no', 'acceder', 'campus', 'virtual']


## 6 · NLTK con el mismo mensaje
Usamos **WordPunctTokenizer** (segmentación por palabras y signos) y **SnowballStemmer de español** (stemming). No activamos un modelo POS o NER. Tampoco usamos `WordNetLemmatizer` de inglés sobre español.

La diferencia de segmentación entre spaCy y NLTK refleja estos tokenizadores concretos. El stemming de `vino` no cambia entre las dos frases porque esta función solo recibe la palabra, no su contexto. Un stem no equivale a un lema.

In [13]:
import nltk
from nltk.tokenize import WordPunctTokenizer
from nltk.stem.snowball import SnowballStemmer
tokenizador_nltk = WordPunctTokenizer()
stemmer = SnowballStemmer('spanish')
tokens_nltk = tokenizador_nltk.tokenize(TEXTO)
print('NLTK',nltk.__version__)
print('Tokens NLTK:',tokens_nltk)
print('Tokens spaCy:',[t.text for t in doc])
for palabra in ['vino','matrícula','acceder','recuperar','cantábamos']:
    print(palabra,'stem:',stemmer.stem(palabra))
print('En ambas frases vino produce el mismo stem:',stemmer.stem('vino'))

NLTK 3.9.2
Tokens NLTK: ['Hola', ',', 'soy', 'Ana', ',', 'alumna', 'de', 'UNIR', 'en', 'Madrid', '.', 'Desde', 'ayer', 'no', 'puedo', 'acceder', 'al', 'campus', 'virtual', '.', 'No', 'quiero', 'anular', 'mi', 'matrícula', ',', 'solo', 'recuperar', 'el', 'acceso', '.', 'Ticket', 'INC', '-', '204', '.']
Tokens spaCy: ['Hola', ',', 'soy', 'Ana', ',', 'alumna', 'de', 'UNIR', 'en', 'Madrid', '.', 'Desde', 'ayer', 'no', 'puedo', 'acceder', 'al', 'campus', 'virtual', '.', 'No', 'quiero', 'anular', 'mi', 'matrícula', ',', 'solo', 'recuperar', 'el', 'acceso', '.', 'Ticket', 'INC-204', '.']
vino stem: vin
matrícula stem: matricul
acceder stem: acced
recuperar stem: recuper
cantábamos stem: cant
En ambas frases vino produce el mismo stem: vin


## 7 · Prompts para IA generativa
Copia el primer prompt en la IA que sueles utilizar.
La comparación A pide anotaciones equivalentes.
La comparación B añade una tarea de intención que no hemos programado en los pipelines locales.


In [15]:
PROMPT_ANOTACION = f"""Analiza este texto en español. Devuelve JSON con:
1. tokens: lista de objetos con texto, lema y POS usando etiquetas Universal POS.
2. entidades: texto literal, tipo y evidencia literal.
No corrijas ni reescribas el texto. No añadas entidades que no aparezcan.
Texto: {TEXTO}
"""
PROMPT_INTENCION = f"""Actúas como analista del servicio técnico de alumnos de UNIR.
Usa SOLO el texto. Devuelve un objeto JSON sin markdown con estas claves:
problema (string), intencion (recuperar_acceso, anular_matricula, otra o indeterminada),
quiere_anular_matricula (true, false o null), evidencia_intencion (cita literal exacta),
ticket (string o null), desde_cuando (string o null), causa_tecnica (string o null),
urgencia (string o null), datos_que_faltan (lista de strings).
Respeta la negación. Usa null cuando no conste. No inventes políticas, enlaces o pasos técnicos.
Texto: {TEXTO}
"""
print('A · Anotación comparable\n',PROMPT_ANOTACION)
print('B · Intención del ticket\n',PROMPT_INTENCION)

A · Anotación comparable
 Analiza este texto en español. Devuelve JSON con:
1. tokens: lista de objetos con texto, lema y POS usando etiquetas Universal POS.
2. entidades: texto literal, tipo y evidencia literal.
No corrijas ni reescribas el texto. No añadas entidades que no aparezcan.
Texto: Hola, soy Ana, alumna de UNIR en Madrid. Desde ayer no puedo acceder al campus virtual. No quiero anular mi matrícula, solo recuperar el acceso. Ticket INC-204.

B · Intención del ticket
 Actúas como analista del servicio técnico de alumnos de UNIR.
Usa SOLO el texto. Devuelve un objeto JSON sin markdown con estas claves:
problema (string), intencion (recuperar_acceso, anular_matricula, otra o indeterminada),
quiere_anular_matricula (true, false o null), evidencia_intencion (cita literal exacta),
ticket (string o null), desde_cuando (string o null), causa_tecnica (string o null),
urgencia (string o null), datos_que_faltan (lista de strings).
Respeta la negación. Usa null cuando no conste. No inven

## Referencias
Recuerda:
El texto completo permite inspeccionar lo que nuestras representaciones pierden. Una biblioteca puede combinar reglas y modelos entrenados. Un modelo generativo puede realizar varias tareas mediante instrucciones, pero hay que comprobar su salida con evidencia del texto.

**Fuentes recomendadas para consultar:**
- https://spacy.io/usage/linguistic-features
- https://spacy.io/usage/rule-based-matching#entityruler
- https://www.nltk.org/api/nltk.tokenize.html
- https://www.nltk.org/api/nltk.stem.snowball.html
- https://www.nltk.org/_modules/nltk/tag.html
- https://huggingface.co/docs/tokenizers/pipeline

